# Tirana City Portrait
_Ongoing work_

This notebook assembles a compact website-ready preview of the Tirana case study.  
It combines a small archival synthesis table, an OSM-oriented interpretation layer, and a schematic map preview.

## 1. Base Map Layers

The base map stack used in the broader analysis includes:

- OpenStreetMap
- Overture
- GDE

## 2. Statistical Layers

Selected statistical layers used in the broader workflow:

- Global Exposure Model
- GHSL: Shape to Point
- Census


## 3. Archival Data

The table below summarizes selected areas and neighborhood types in Tirana discussed in archival and morphological literature. Streets are listed only where relatively precise references are available.


In [2]:
{
  "tags": ["hide-input"]
}


import pandas as pd
from IPython.display import display, HTML

rows = [
    [
        "Historic Ottoman Tirana core",
        "pre-1920 → early 20th c.",
        ["Pazari i Ri", "Et'hem Bey Mosque surroundings", "Rruga Barrikadave"],
        ["Courtyard houses", "Bazaar fabric", "Small plot mixed-use", "Commercial strips"],
        ["Ottoman urban structure", "Tirana becomes capital (1920)"],
        [],
    ],
    [
        "Monarchy-era planned capital",
        "1920–1940",
        ["Skanderbeg Square", "Bulevardi Dëshmorët e Kombit", "Rruga e Durrësit"],
        ["Institutional ensembles", "Perimeter blocks (neoclassical / rationalist)", "Monumental civic complexes"],
        ["Capital consolidation", "Italian planning influence"],
        ["Armando Brasini", "Florestano Di Fausto", "Gherardo Bosio"],
    ],
    [
        "Socialist centre zone",
        "1945–1990",
        ["Skanderbeg Square", "Bulevardi Stalin / Dëshmorët e Kombit", "Lana axis"],
        ["Palace of Culture", "National Historical Museum", "Central institutions", "State service buildings"],
        ["Socialist reconstruction", "Centralized planning", "Ideological monumentalization"],
        [],
    ],
    [
        "Socialist housing estates",
        "1950s–1980s",
        ["Rruga e Kavajës", "Rruga e Dibrës", "Rruga Myslym Shyri"],
        ["Prefabricated apartment blocks", "Standardized slab housing", "Neighborhood units"],
        ["State housing provision", "Industrialized construction"],
        [],
    ],
    [
        "Shallvaret",
        "late socialist → post-socialist transition",
        [],
        ["Mid-rise apartment blocks", "Infill development"],
        ["Late socialist housing intensification", "Post-1990 transformation"],
        [],
    ],
    [
        "Kombinati",
        "socialist industrialization → post-1990",
        ["Kombinati axis"],
        ["Worker housing", "Industrial-residential ensembles"],
        ["Industrial expansion", "Post-socialist decline / transformation"],
        [],
    ],
    [
        "Urban edges / peripheral expansion",
        "1990s → present",
        [],
        ["Informal settlements", "Self-built housing", "Recent mixed-density expansion"],
        ["Mass rural-urban migration", "Informal urban growth", "Regularization processes"],
        [],
    ],
]

columns = [
    "Area / Neighborhood",
    "Period of Evolution",
    "Streets / Axes",
    "Building Complexes",
    "Socio-Political Events",
    "Actors",
]

df = pd.DataFrame(rows, columns=columns)

def to_html_list(items):
    if not items:
        return "<span class='empty'>—</span>"
    return "<ul class='cell-list'>" + "".join(f"<li>{item}</li>" for item in items) + "</ul>"

df_display = df.copy()
for col in ["Streets / Axes", "Building Complexes", "Socio-Political Events", "Actors"]:
    df_display[col] = df_display[col].apply(to_html_list)

styled_html = """
<style>
.tirana-table {
  width: 100%;
  border-collapse: collapse;
  font-size: 0.92rem;
  line-height: 1.45;
}
.tirana-table th, .tirana-table td {
  border: 1px solid #d9d9d9;
  padding: 0.65rem 0.75rem;
  vertical-align: top !important;
  text-align: left !important;
}
.tirana-table th {
  background: #f5f5f5;
  font-weight: 600;
}
.tirana-table .cell-list {
  margin: 0;
  padding-left: 1.15rem;
  list-style-position: outside;
}
.tirana-table .cell-list li {
  margin: 0 0 0.18rem 0;
}
.tirana-table .empty {
  color: #777;
}
</style>
""" + df_display.to_html(classes="tirana-table", escape=False, index=False)

display(HTML(styled_html))


Area / Neighborhood,Period of Evolution,Streets / Axes,Building Complexes,Socio-Political Events,Actors
Historic Ottoman Tirana core,pre-1920 → early 20th c.,Pazari i RiEt'hem Bey Mosque surroundingsRruga Barrikadave,Courtyard housesBazaar fabricSmall plot mixed-useCommercial strips,Ottoman urban structureTirana becomes capital (1920),—
Monarchy-era planned capital,1920–1940,Skanderbeg SquareBulevardi Dëshmorët e KombitRruga e Durrësit,Institutional ensemblesPerimeter blocks (neoclassical / rationalist)Monumental civic complexes,Capital consolidationItalian planning influence,Armando BrasiniFlorestano Di FaustoGherardo Bosio
Socialist centre zone,1945–1990,Skanderbeg SquareBulevardi Stalin / Dëshmorët e KombitLana axis,Palace of CultureNational Historical MuseumCentral institutionsState service buildings,Socialist reconstructionCentralized planningIdeological monumentalization,—
Socialist housing estates,1950s–1980s,Rruga e KavajësRruga e DibrësRruga Myslym Shyri,Prefabricated apartment blocksStandardized slab housingNeighborhood units,State housing provisionIndustrialized construction,—
Shallvaret,late socialist → post-socialist transition,—,Mid-rise apartment blocksInfill development,Late socialist housing intensificationPost-1990 transformation,—
Kombinati,socialist industrialization → post-1990,Kombinati axis,Worker housingIndustrial-residential ensembles,Industrial expansionPost-socialist decline / transformation,—
Urban edges / peripheral expansion,1990s → present,—,Informal settlementsSelf-built housingRecent mixed-density expansion,Mass rural-urban migrationInformal urban growthRegularization processes,—


## 4. OSM-oriented interpretation

This is not a strict data extraction step. It is a lightweight translation layer that connects archival building categories to possible OSM-facing tags or lookup ideas.


In [3]:

import pandas as pd
from IPython.display import display

osm_mapping = {
    "Courtyard houses": {"building": "residential"},
    "Bazaar fabric": {"landuse": "commercial"},
    "Commercial strips": {"shop": "*"},
    "Institutional ensembles": {"building": "government"},
    "Monumental civic complexes": {"building": "public"},
    "Prefabricated apartment blocks": {"building": "apartments"},
    "Worker housing": {"building": "residential"},
    "Informal settlements": {"building": "residential"},
    "Industrial-residential ensembles": {"landuse": "industrial"},
}

sample = pd.DataFrame(
    [
        {
            "Archival Category": category,
            "Possible OSM Tag": ", ".join(f"{k}={v}" for k, v in tags.items()),
        }
        for category, tags in osm_mapping.items()
    ]
)

display(sample)


,Archival Category,Possible OSM Tag
0,Courtyard houses,building=residential
1,Bazaar fabric,landuse=commercial
2,Commercial strips,shop=*
3,Institutional ensembles,building=government
4,Monumental civic complexes,building=public
5,Prefabricated apartment blocks,building=apartments
6,Worker housing,building=residential
7,Informal settlements,building=residential
8,Industrial-residential ensembles,landuse=industrial


## 5. Map preview

The map below is a schematic preview. It is meant for website communication rather than precise boundary mapping: selected streets are represented as curated editorial itineraries, and circles act as simple proxies for the relative prominence of selected urban areas.


In [ ]:
from pathlib import Path
import pandas as pd
import folium
import geopandas as gpd
import numpy as np

from shapely import wkt
from IPython.display import display

# =========================================================
# 1) INPUT FILES
# =========================================================
CSV_STREETS_PATH = r"C:\Users\ReDI\Documents\GitHub\2026_GREAM\training\20260604-01_SummerSchool_docomomo\CityPortraits\docs\assets\data\case_studies\tirana\streets.csv"
CSV_SPITALET_PATH = r"C:\Users\ReDI\Documents\GitHub\2026_GREAM\training\20260604-01_SummerSchool_docomomo\CityPortraits\docs\assets\data\case_studies\tirana\spitalet.csv"
CSV_HOUSING_PATH = r"C:\Users\ReDI\Documents\GitHub\2026_GREAM\training\20260604-01_SummerSchool_docomomo\CityPortraits\docs\assets\data\case_studies\tirana\housing.csv"
CSV_INDUSTRI_PATH = r"C:\Users\ReDI\Documents\GitHub\2026_GREAM\training\20260604-01_SummerSchool_docomomo\CityPortraits\docs\assets\data\case_studies\tirana\industri.csv"

# =========================================================
# 2) THEMATIC STYLES
# =========================================================
theme_colors = {
    "Spitalet": "#d73027",
    "Housing": "#1a9850",
    "Industri": "#4575b4",
}

# =========================================================
# 3) HELPERS
# =========================================================
def load_wkt_csv(csv_path, expected_geom_types=None):
    df = pd.read_csv(csv_path)

    if "WKT" not in df.columns:
        raise ValueError(f"{csv_path} must contain a 'WKT' column.")

    if "name" not in df.columns:
        df["name"] = [f"Feature_{i+1}" for i in range(len(df))]

    df["geometry"] = df["WKT"].apply(wkt.loads)
    gdf = gpd.GeoDataFrame(df.copy(), geometry="geometry", crs="EPSG:4326")

    if expected_geom_types is not None:
        gdf = gdf[gdf.geometry.geom_type.isin(expected_geom_types)].copy()

    return gdf


def area_to_radius_m(area_m2):
    return max(120, min(520, np.sqrt(area_m2) * 1.35))


def point_layer_to_bubble_row(gdf, layer_name, color):
    if gdf.empty:
        return None

    gdf_metric = gdf.to_crs(3857)
    union_geom = gdf_metric.geometry.union_all()

    if union_geom is None or union_geom.is_empty:
        return None

    centroid_metric = union_geom.centroid
    centroid_geo = gpd.GeoSeries([centroid_metric], crs=3857).to_crs(4326).iloc[0]

    # dummy proxy area from number of points
    area_m2 = max(12000, len(gdf) * 3500.0)

    return {
        "name": layer_name,
        "lat": centroid_geo.y,
        "lon": centroid_geo.x,
        "area_m2": area_m2,
        "color": color,
        "count": len(gdf),
    }


def add_streets_to_map(m, streets_gdf):
    for _, row in streets_gdf.iterrows():
        geom = row.geometry
        name = row.get("name", "Street")

        if geom is None or geom.is_empty:
            continue

        if geom.geom_type == "LineString":
            parts = [geom]
        elif geom.geom_type == "MultiLineString":
            parts = list(geom.geoms)
        else:
            continue

        for part in parts:
            coords = [(lat, lon) for lon, lat in part.coords]

            # soft white halo
            folium.PolyLine(
                coords,
                color="#ffffff",
                weight=8,
                opacity=0.78,
                tooltip=name,
            ).add_to(m)

            # main street line
            folium.PolyLine(
                coords,
                color="#4a4a4a",
                weight=5,
                opacity=0.97,
                tooltip=name,
            ).add_to(m)


def add_point_layer_to_map(m, gdf, layer_name, color):
    for _, row in gdf.iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue
        if geom.geom_type != "Point":
            continue

        name = row.get("name", layer_name)

        folium.CircleMarker(
            location=[geom.y, geom.x],
            radius=5,
            color=color,
            weight=1,
            fill=True,
            fill_color=color,
            fill_opacity=0.95,
            popup=f"{layer_name}<br>{name}",
            tooltip=name,
        ).add_to(m)


def add_bubble_to_map(m, bubble_row):
    if bubble_row is None:
        return

    folium.Circle(
        location=[bubble_row["lat"], bubble_row["lon"]],
        radius=area_to_radius_m(bubble_row["area_m2"]),
        color=bubble_row["color"],
        weight=2,
        fill=True,
        fill_color=bubble_row["color"],
        fill_opacity=0.16,
        tooltip=f"{bubble_row['name']} ({bubble_row['count']} points)",
    ).add_to(m)

    folium.CircleMarker(
        location=[bubble_row["lat"], bubble_row["lon"]],
        radius=4,
        color=bubble_row["color"],
        weight=1,
        fill=True,
        fill_color=bubble_row["color"],
        fill_opacity=1.0,
        popup=(
            f"{bubble_row['name']}"
            f"<br>Count: {bubble_row['count']}"
            f"<br>Dummy area proxy: {bubble_row['area_m2']:,.0f} m²"
        ),
    ).add_to(m)


# =========================================================
# 4) LOAD DATA
# =========================================================
streets_gdf = load_wkt_csv(
    CSV_STREETS_PATH,
    expected_geom_types=["LineString", "MultiLineString"]
)

spitalet_gdf = load_wkt_csv(
    CSV_SPITALET_PATH,
    expected_geom_types=["Point", "MultiPoint"]
)

housing_gdf = load_wkt_csv(
    CSV_HOUSING_PATH,
    expected_geom_types=["Point", "MultiPoint"]
)

industri_gdf = load_wkt_csv(
    CSV_INDUSTRI_PATH,
    expected_geom_types=["Point", "MultiPoint"]
)

print("Loaded streets:", len(streets_gdf))
print("Loaded Spitalet points:", len(spitalet_gdf))
print("Loaded Housing points:", len(housing_gdf))
print("Loaded Industri points:", len(industri_gdf))

# =========================================================
# 5) BUBBLE DEFINITIONS
# =========================================================
bubble_rows = [
    point_layer_to_bubble_row(spitalet_gdf, "Spitalet", theme_colors["Spitalet"]),
    point_layer_to_bubble_row(housing_gdf, "Housing", theme_colors["Housing"]),
    point_layer_to_bubble_row(industri_gdf, "Industri", theme_colors["Industri"]),
]
bubble_rows = [x for x in bubble_rows if x is not None]

# =========================================================
# 6) MAP
# =========================================================
m = folium.Map(
    location=[41.3275, 19.8187],
    zoom_start=13,
    tiles="CartoDB positron",
    control_scale=True
)

# Streets
add_streets_to_map(m, streets_gdf)

# Thematic points
add_point_layer_to_map(m, spitalet_gdf, "Spitalet", theme_colors["Spitalet"])
add_point_layer_to_map(m, housing_gdf, "Housing", theme_colors["Housing"])
add_point_layer_to_map(m, industri_gdf, "Industri", theme_colors["Industri"])

# Bubbles
for bubble_row in bubble_rows:
    add_bubble_to_map(m, bubble_row)

# =========================================================
# 7) FIT BOUNDS
# =========================================================
all_bounds = []

for rruga_dibres_gdf in [streets_gdf, spitalet_gdf, housing_gdf, industri_gdf]:
    if not rruga_dibres_gdf.empty:
        all_bounds.append(rruga_dibres_gdf.total_bounds)

if all_bounds:
    minx = min(b[0] for b in all_bounds)
    miny = min(b[1] for b in all_bounds)
    maxx = max(b[2] for b in all_bounds)
    maxy = max(b[3] for b in all_bounds)
    m.fit_bounds([[miny, minx], [maxy, maxx]])

# =========================================================
# 8) LEGEND
# =========================================================
legend_items = "".join(
    f"""
    <div style="display:flex; align-items:center; margin-bottom:6px;">
        <span style="
            display:inline-block;
            width:12px;
            height:12px;
            border-radius:50%;
            background:{c};
            margin-right:8px;
        "></span>
        <span>{n}</span>
    </div>
    """
    for n, c in theme_colors.items()
)

legend_html = f"""
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    z-index: 9999;
    background: white;
    padding: 12px 14px;
    border: 1px solid #d0d0d0;
    border-radius: 10px;
    font-size: 13px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.12);
">
    <div style="font-weight:600; margin-bottom:8px;">Thematic layers</div>
    {legend_items}
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

display(m)

Loaded streets: 3
Loaded Spitalet points: 10
Loaded Housing points: 10
Loaded Industri points: 2


# Filter GIS Data

In [1]:
# Imports
from pathlib import Path
import pandas as pd
import folium
import geopandas as gpd
import numpy as np
from shapely import wkt
from IPython.display import display

In [2]:
# Paths
BASE_DIR = Path(r"C:\Users\ReDI\Documents\GitHub\2026_GREAM\training\20260604-01_SummerSchool_docomomo\CityPortraits\docs\assets\data\case_studies\tirana")
GEOJSON_PATH = BASE_DIR / "maps" / "fixed.geojson"

In [3]:
# Load GeoJSON
rruga_dibres_gdf = gpd.read_file(GEOJSON_PATH)



rruga_dibres_gdf = rruga_dibres_gdf.convert_dtypes()

for col in rruga_dibres_gdf.columns:
    if rruga_dibres_gdf[col].dtype == "datetime64[ns]":
        rruga_dibres_gdf[col] = rruga_dibres_gdf[col].astype(str)

if rruga_dibres_gdf.crs is None:
    rruga_dibres_gdf = rruga_dibres_gdf.set_crs("EPSG:4326")
else:
    rruga_dibres_gdf = rruga_dibres_gdf.to_crs("EPSG:4326")

rruga_dibres_gdf = rruga_dibres_gdf[rruga_dibres_gdf.geometry.notna() & ~rruga_dibres_gdf.geometry.is_empty]
print(len(rruga_dibres_gdf))

Skipping field collection_times: unsupported OGR type: 10


35404


In [ ]:
# HELPER — load_geojson

import geopandas as gpd

def load_geojson(path):
    rruga_dibres_gdf = gpd.read_file(path)

    # Ensure CRS
    if rruga_dibres_gdf.crs is None:
        rruga_dibres_gdf = rruga_dibres_gdf.set_crs("EPSG:4326")
    else:
        rruga_dibres_gdf = rruga_dibres_gdf.to_crs("EPSG:4326")

    # Remove bad geometries
    rruga_dibres_gdf = rruga_dibres_gdf[
        rruga_dibres_gdf.geometry.notna() & ~rruga_dibres_gdf.geometry.is_empty
    ].copy()

    return rruga_dibres_gdf

In [5]:
rruga_dibres_gdf = load_geojson(GEOJSON_PATH)

# 🔴 HARD CLEAN for Folium compatibility
rruga_dibres_gdf = rruga_dibres_gdf.copy()

for col in rruga_dibres_gdf.columns:
    # convert datetime → string
    if "datetime" in str(rruga_dibres_gdf[col].dtype):
        rruga_dibres_gdf[col] = rruga_dibres_gdf[col].astype(str)

    # convert pandas types → python native
    if str(rruga_dibres_gdf[col].dtype) == "object":
        rruga_dibres_gdf[col] = rruga_dibres_gdf[col].astype(str)

Skipping field collection_times: unsupported OGR type: 10


In [6]:
# Make geometry lighter for Folium
rruga_dibres_gdf_light = rruga_dibres_gdf.copy()

# keep only geometry to avoid heavy attributes
rruga_dibres_gdf_light = rruga_dibres_gdf_light[["geometry"]]

# simplify geometry slightly
rruga_dibres_gdf_light["geometry"] = (
    rruga_dibres_gdf_light
    .to_crs(3857)
    .geometry
    .simplify(1.5)
    .to_crs(4326)
)

In [7]:
# TEST — reduce GeoJSON for fast preview

# keep only geometry + limit features
rruga_dibres_gdf_light = rruga_dibres_gdf[["geometry"]].head(200)

print("Using subset:", len(rruga_dibres_gdf_light))

Using subset: 200


In [8]:
# Create map
m = folium.Map(location=[41.3275, 19.8187], zoom_start=13, tiles="CartoDB positron")

folium.GeoJson(
    rruga_dibres_gdf_light,
    name="GeoJSON Layer",
    style_function=lambda f: {"color": "#ff7800", "weight": 2, "fillOpacity": 0.25}
).add_to(m)

folium.LayerControl().add_to(m)

display(m)

In [ ]:
# # Create map
# m = folium.Map(location=[41.3275, 19.8187], zoom_start=13, tiles="CartoDB positron")

# folium.GeoJson(
#     rruga_dibres_gdf,
#     name="GeoJSON Layer",
#     style_function=lambda f: {"color": "#ff7800", "weight": 2, "fillOpacity": 0.25}
# ).add_to(m)

# folium.LayerControl().add_to(m)

# display(m)

## 6. Clustering

- Rule-based / event-based suggestions
- Algorithm: spatial interpolation

## 7. Building Stock Scenarios

- Algorithm: filter based on archival data
- Redistribute
